# 30장 · 평가자–개선자 · 작성, 평가, 수정을 세 호출로 나누기

평가자–개선자는 초안을 평가하는 호출과 고치는 호출을 나누는 기법입니다. 평가 의견을 다음 호출에 직접 전달해야 수정할 기준이 생깁니다.

## 초안과 평가 의견을 수정 요청에 넣기

초안을 고칠 때는 어떤 글을 고칠지와 어디를 손볼지를 함께 전달합니다. 첫 호출에서 수정 의견을 얻고, 두 번째 호출에 초안과 그 의견을 넣습니다.

`ask(지시, 입력)`은 OpenAI SDK로 요청하고 답을 문자열로 돌려주는 수업용 함수입니다. 수업 커널에서 먼저 아래 셀로 함수를 정의하세요. 새 커널에서 openai를 찾지 못하면 `%pip install openai`를 실행하세요.

In [ ]:
import os
from getpass import getpass
from openai import OpenAI

model = os.environ.get("OPENAI_MODEL", "gpt-4.1-mini")
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API 키: ")

client = OpenAI(timeout=45, max_retries=1)  # 8장에서 준비한 환경 변수 OPENAI_API_KEY를 읽습니다.

def ask(instruction, material):
    response = client.responses.create(
        model=model, instructions=instruction, input=material, store=False
    )
    return response.output_text

In [ ]:
draft = "봄. 따뜻함."
feedback = ask("이 글을 한 문장으로 만들 방법을 제안하세요.", draft)
revised = ask("의견을 반영해 글을 고치세요.", draft + "\n의견: " + feedback)
print(feedback)
print(revised)

먼저 feedback에 담긴 의견을 읽고, 그 아래 revised에서 실제로 고친 표현을 찾아보세요. 두 번째 호출의 입력에서 draft를 빼면 모델에는 고칠 글 대신 의견만 전달됩니다.

## 응용 실습 · 평가 의견으로 초안 고치기

### 작업별 답과 근거를 함께 읽기

29장의 워커들이 같은 문의에 서로 다른 부분 답을 썼습니다. 어떤 답이 어느 요구를 해결했는지 먼저 대조한 뒤 마지막 실습에서 하나의 회원 안내로 합칩니다. 앞 장을 건너뛰면 수업 묶음에 들어 있는 복귀 예시를 읽습니다. 첫 출력에 자료의 출처가 표시됩니다.

In [ ]:
from pathlib import Path
import json
import sys

root = next(folder for folder in (Path.cwd(), *Path.cwd().parents)
            if (folder / "pre-mid/documents/workflow").is_dir())
sys.path.insert(0, str(root / "pre-mid/documents/workflow"))
from workbook_files import read_previous, save_record

previous = read_previous(root, 29)
for part in previous["parts"]:
    print(part["근거 구절"], "→", part["답"])

교재 복귀 예시 · chapter29.json
언제까지 어디에 → 6월 11일 정오까지 원래 신청 게시글에 연장 요청 댓글을 남깁니다.
댓글만 쓰면 → 담당자의 승인 댓글을 확인해야 합니다.


## 응용에 사용할 자료 열기

실습 묶음의 압축을 풀고 노트북과 `pre-mid` 폴더를 나란히 두세요. 다음 셀은 대여 규정 파일을 읽어 `rules`에 저장합니다. 응용에서도 앞에서 정의한 `ask`를 사용합니다.

In [ ]:
from pathlib import Path

rules = (next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pre-mid/documents/sources/loan-rules.md").is_file()) / "pre-mid/documents/sources/loan-rules.md").read_text(encoding="utf-8")

In [ ]:
question = '6월 10일에 A형을 직접 받았어요. 반납을 하루 미루려면 언제까지 어디에 남기면 되나요? 댓글만 쓰면 연장이 끝나는지도 궁금해요.'
print(question)

## 평가 지시에 넣을 통과 조건 정하기

검토할 때 무엇이 빠졌으면 수정을 요구할지 먼저 정합니다. 이번 예제에서는 연장 요청의 기한과 위치, 담당자 승인 필요 여부를 초안에서 찾습니다. `judge_instruction`은 평가 의견을 쓰는 지시이고, 그 뒤의 `revise_instruction`은 초안을 다시 쓰는 지시입니다.

## 같은 문의의 두 부분 답을 취합하기

29장에서 만들었던 하위 답 기록의 형태를 다시 사용합니다. 아래 두 문장은 사람이 규정을 보고 미리 쓴 수업 입력입니다. 두 사실을 각각 따로 읽으면 맞습니다. 하나의 안내로 합칠 때 시각·위치·승인 주체가 어떻게 남는지 확인하세요.

In [ ]:
parts = [
    "언제 어디에? → 원래 반납일 6월 11일 정오까지 원래 신청 게시글에 연장 요청 댓글을 남긴다.",
    "댓글만 쓰면 완료? → 담당자가 승인 댓글을 단 뒤에 한 번, 하루까지 연장된다.",
]
material = question + "\n부분 답\n" + "\n".join(parts)

In [ ]:
merged = ask("두 부분 답을 회원에게 보낼 하나의 짧은 안내로 합치세요. 두 질문에 모두 답하세요.", material)
print(merged)

“댓글을 남기면 됩니다”에서 끝난다면 요청 절차와 예약 상태를 혼동할 수 있습니다. 원문에서 명확했던 승인 조건을 합치기 전과 후에 각각 찾아보세요. 이미 잘 합쳐졌다면 그 버전은 그대로 보존합니다.

## 고쳐야 할 초안을 따로 넣어 진단하기

모델이 잘 쓴 답을 일부러 틀렸다고 평가할 필요는 없습니다. 아래는 승인 조건을 의도적으로 뺀 집필자 초안입니다. 평가자가 개선을 요구할 때 “조금 더 자세히”처럼 행동으로 옮기기 어려운 말만 주는지, 규정의 어느 조건이 빠졌는지 짚는지 확인하세요.

In [ ]:
draft = "6월 11일 정오까지 원래 신청 게시글에 하루 연장을 요청하는 댓글을 남겨 주세요."
judge_instruction = '규정·회원 문의·초안을 대조하세요. 회원이 물은 모든 행동에 필요한 조건이 있고\n규정과 일치하면 첫 줄에 통과, 아니면 수정이라고 쓰세요. 다음 줄에는\n원문 조건과 초안에서 빠지거나 잘못된 구절을 짝지어 한두 문장으로 설명하세요.\n문의에서 묻지 않은 주변 규정을 추가하라고 요구하지 마세요.'

평가자와 개선자에 같은 `ask` 함수를 쓰되, 호출마다 읽을 자료를 바꿉니다. 평가자는 규정·원래 질문·초안을 받아 판정과 이유를 글로 돌려줍니다. 개선자는 그 글까지 붙인 `revision_input`을 받아 새 초안을 씁니다. 평가 의견을 화면에만 출력하면 다음 요청에는 아무것도 추가되지 않습니다. 아래의 `feedback`이 어느 문자열에 들어가는지 확인하세요.

In [ ]:
review_input = rules + "\n문의\n" + question + "\n초안\n" + draft
feedback = ask(judge_instruction, review_input)
print(feedback)

수정 판단은 R4의 승인 댓글을 가리켜야 합니다. 평가자가 지적했다는 사실만으로 지적이 옳다고 보지 마세요. 승인 시각까지 정하라고 했다면 규정에 해당 시각이 있는지 되돌아가 읽어야 합니다.

## revision_input에 초안과 feedback을 함께 넣기

`review_input`에는 평가자가 읽은 규정·질문·초안이 들어 있습니다. 다음 줄은 그 문자열 뒤에 `feedback`을 붙입니다. `ask(revise_instruction, revision_input)`에서 두 번째 인자를 이전 `review_input`으로 되돌리면 작성자는 방금 받은 의견을 볼 수 없습니다.

In [ ]:
revise_instruction = '회원에게 보낼 초안을 고치세요. 검토 의견을 규정과 대조하고, 맞는 조건은 유지하세요. 설명 없이 수정한 안내만 쓰세요.'
revision_input = review_input + "\n검토 의견\n" + feedback
revised = ask(revise_instruction, revision_input)
print(revised)

수정본에서 새로 들어간 승인 조건 하나와 계속 남아야 하는 댓글 위치 하나를 짚으세요. 첫 답과 같은 표현을 보존할 필요는 없지만 행동, 담당자, 기한은 달라지면 안 됩니다.

## 평가 의견이 수정으로 이어지는 예 읽기

**집필자가 쓴 평가 예시**　수정. R4의 “담당자가 승인 댓글을 달아야”가 초안에는 없습니다. 현재 문장만 읽으면 댓글로 연장이 끝난다고 생각할 수 있습니다.

**수정한 문장 예시**　6월 11일 정오까지 원래 신청 게시글에 하루 연장을 요청하는 댓글을 남겨 주세요. 담당자가 승인 댓글을 달아야 반납일이 바뀝니다.

앞 문장에서 지켜 낸 조건은 원래 게시글과 정오입니다. 뒤 문장에서 추가한 조건은 승인 주체입니다. 자기 결과에서 새로 들어간 문구와 사라진 문구를 둘 다 찾아보세요.

## 부분 답의 동사가 바뀌는 순간 찾기

“댓글을 남겨 연장을 요청합니다”와 “댓글을 남겨 기간을 연장합니다”를 비교하세요. 두 문장의 차이는 단어 하나에 가깝지만 두 번째 문장은 누가 승인하는지 사라졌습니다. 취합을 평가할 때 시간·장소 같은 명사뿐 아니라 요청한다, 승인한다, 바뀐다는 동사를 표시하세요. 회원에게 다음 행동을 알려 주는 문장과 그 뒤 일어나는 결과를 설명하는 문장을 따로 쓴 뒤 합쳐 보면 승인까지 완료된 듯한 표현을 발견하기 쉽습니다.

준비물은 실습 묶음의 `pre-mid/documents/sources/loan-rules.md`와 `pre-mid/documents/questions/loan-inquiries.md`입니다. 짝과 검토할 기록지는 `pre-mid/documents/workflow/workshop.md`에 있습니다.

## 실습 · 좋은 문장과 빠진 조건을 함께 넘기기

개인 실습 90분. 새 초안 “6월 11일 오후 5시 전에 동아리 게시판에 글을 새로 써 주세요. 승인이 필요해요.”를 검사하세요. 승인 조건은 유지하고 특정 게시글과 시각을 정확히 고쳐야 합니다. 첫 답, 평가 의견, 수정 입력, 마지막 답의 해당 구절을 나란히 놓고 한 조건씩 짝지으세요. 짝이 규정으로 확인할 수 없는 수정 제안은 채택하지 마세요.

In [ ]:
candidate = "6월 11일 오후 5시 전에 동아리 게시판에 글을 새로 써 주세요. 승인이 필요해요."
# 새 candidate를 사용해 검토와 수정 요청을 이어 보세요.

<details class="notebook-answer"><summary>모범 풀이와 대조 기준</summary>
<pre><code>candidate = &quot;6월 11일 오후 5시 전에 동아리 게시판에 글을 새로 써 주세요. 승인이 필요해요.&quot;
inputs = rules + &quot;\n&quot; + question + &quot;\n초안\n&quot; + candidate
comments = ask(judge_instruction, inputs)
print(comments)
print(ask(revise_instruction, inputs + &quot;\n검토 의견\n&quot; + comments))</code></pre>
<p>“오후 5시”는 원 기기의 반납 기한이고 연장 댓글 마감은 정오입니다. “글을 새로”도 기존 신청 게시글의 댓글로 고쳐야 합니다. 수정 예시는 “6월 11일 정오까지 원래 신청 게시글에 연장 요청 댓글을 남기세요. 담당자가 승인 댓글을 달아야 연장이 확정됩니다.”입니다.</p>
</details>

## 내 워커가 작성한 답으로 수정본 만들기

연장 문의의 예시에서 연습한 평가·수정 절차를 앞 장의 실제 문의에 적용합니다. 평가자에게 원문과 부분 답을 함께 주고, 수정 뒤에도 회원이 할 행동과 담당자가 할 행동을 구별할 수 있는지 읽어 보세요.

In [ ]:
write_instruction = "문의의 각 요구에 답하는 짧은 안내를 쓰세요."
original = previous["question"]
worker_text = "\n".join(part["답"] for part in previous["parts"])
combined_draft = ask(write_instruction, original + "\n부분 답\n" + worker_text)
context = rules + "\n원문\n" + original + "\n초안\n" + combined_draft
review = ask(judge_instruction, context)
polished = ask(revise_instruction, context + "\n의견\n" + review)
record = {"question": original, "draft": combined_draft, "feedback": review, "revised": polished}
save_record(root, 30, record)

검토자가 찾아낸 조건이 수정본의 어느 문장에 들어갔는지 비교하세요. 다음 셀에서 의견과 수정본을 따로 읽습니다. 실시간 모델을 실행하지 않았으면 아래 셀만 골라 실행해 교재 예시의 자료 구조를 읽을 수 있습니다. 첫 줄이 파일의 출처입니다.

In [ ]:
from pathlib import Path
import json
import sys

root = next(folder for folder in (Path.cwd(), *Path.cwd().parents)
            if (folder / "pre-mid/documents/workflow").is_dir())
sys.path.insert(0, str(root / "pre-mid/documents/workflow"))
from workbook_files import read_previous, save_record

saved = read_previous(root, 30)
print(saved["feedback"], saved["revised"], sep="\n")

교재 복귀 예시 · chapter30.json
수정
시각과 승인 조건을 보충하세요.
6월 11일 정오까지 원래 신청 게시글에 연장 요청 댓글을 남기세요. 담당자가 승인 댓글을 달아야 반납일이 바뀝니다.


## 다음 작업에 가져갈 기록

실제 수정 요청과 수정된 안내를 보관하세요. 다음 장에서는 첫 답이 이미 충분한 경우와 한 번 고쳐도 미흡한 경우에 각각 언제 멈출지 결정합니다.

[OpenAI Python SDK 사용 예](https://github.com/openai/openai-python)